In [1]:
import os
import sys
import pandas as pd

PROJECT_ROOT = r"C:\sourcecode\Nhom_10_Big_Data"

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from pyspark.sql import functions as F

from src.config.spark_session import get_spark
from src.config.hdfs_config import SUPERSTORE_DATASET
from src.config.schema import SUPERSTORE_SCHEMA

spark = get_spark()
spark.sparkContext.setLogLevel("WARN")

print("Spark đã khởi tạo.")
print("HDFS path:", SUPERSTORE_DATASET)

Spark đã khởi tạo.
HDFS path: hdfs://master:9000/bigdata/superstore/input/G10_dataset.csv


In [2]:
df_final = (
    spark.read
    .option("header", "true")
    .option("dateFormat", "yyyy-MM-dd")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .schema(SUPERSTORE_SCHEMA)
    .csv(SUPERSTORE_DATASET)
)

df_final.cache()

row_count = df_final.count()
col_count = len(df_final.columns)

print("Rows:", row_count)
print("Cols:", col_count)

df_final.printSchema()
df_final.show(5, truncate=False)

Rows: 500000
Cols: 24
root
 |-- Category: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Market: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Priority: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Profit: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Ship_Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Shipping_Cost: double (nullable = true)
 |-- State: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- weeknum: integer (nullable = true)

+--

In [3]:
expected_columns = SUPERSTORE_SCHEMA.fieldNames()
actual_columns = df_final.columns

missing_columns = sorted(list(set(expected_columns) - set(actual_columns)))
extra_columns = sorted(list(set(actual_columns) - set(expected_columns)))

schema_check_df = pd.DataFrame(
    [
        ("Số dòng dữ liệu", row_count),
        ("Số cột dữ liệu", col_count),
        ("Số cột kỳ vọng", len(expected_columns)),
        ("Số cột bị thiếu", len(missing_columns)),
        ("Số cột phát sinh thêm", len(extra_columns)),
    ],
    columns=["Tiêu chí kiểm tra", "Kết quả"]
)

display(schema_check_df)

print("Cột bị thiếu:", missing_columns)
print("Cột phát sinh thêm:", extra_columns)

,Tiêu chí kiểm tra,Kết quả
0,Số dòng dữ liệu,500000
1,Số cột dữ liệu,24
2,Số cột kỳ vọng,24
3,Số cột bị thiếu,0
4,Số cột phát sinh thêm,0


Cột bị thiếu: []
Cột phát sinh thêm: []


In [4]:
missing_exprs = [
    F.sum(
        F.when(
            F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df_final.columns
]

missing_wide = df_final.select(missing_exprs)
missing_counts = missing_wide.first().asDict()

total_missing = sum(missing_counts.values())
columns_with_missing = sum(1 for v in missing_counts.values() if v > 0)

missing_summary_df = pd.DataFrame(
    [
        ("Tổng số ô missing/rỗng", total_missing),
        ("Số cột có missing/rỗng", columns_with_missing),
    ],
    columns=["Tiêu chí kiểm tra", "Kết quả"]
)

display(missing_summary_df)

missing_detail_df = pd.DataFrame(
    [
        (col_name, int(missing_count))
        for col_name, missing_count in missing_counts.items()
        if missing_count > 0
    ],
    columns=["Column", "Missing_Count"]
)

if len(missing_detail_df) > 0:
    display(missing_detail_df.sort_values("Missing_Count", ascending=False))
else:
    print("Không có cột nào bị missing/rỗng.")

,Tiêu chí kiểm tra,Kết quả
0,Tổng số ô missing/rỗng,0
1,Số cột có missing/rỗng,0


Không có cột nào bị missing/rỗng.


In [5]:
total_rows = df_final.count()
distinct_rows = df_final.distinct().count()
duplicate_rows = total_rows - distinct_rows

duplicate_check_df = pd.DataFrame(
    [
        ("Tổng số dòng", total_rows),
        ("Số dòng distinct", distinct_rows),
        ("Số dòng trùng hoàn toàn", duplicate_rows),
    ],
    columns=["Tiêu chí kiểm tra", "Kết quả"]
)

display(duplicate_check_df)

,Tiêu chí kiểm tra,Kết quả
0,Tổng số dòng,500000
1,Số dòng distinct,500000
2,Số dòng trùng hoàn toàn,0


In [6]:
df_check = (
    df_final
    .withColumn("Sales", F.col("Sales").cast("double"))
    .withColumn("Profit", F.col("Profit").cast("double"))
    .withColumn("Quantity", F.col("Quantity").cast("int"))
    .withColumn("Discount", F.col("Discount").cast("double"))
    .withColumn("Shipping_Cost", F.col("Shipping_Cost").cast("double"))
    .withColumn("Year", F.col("Year").cast("int"))
    .withColumn("weeknum", F.col("weeknum").cast("int"))
)

df_check.cache()

df_check.printSchema()

root
 |-- Category: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Market: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Priority: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Profit: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Ship_Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Shipping_Cost: double (nullable = true)
 |-- State: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- weeknum: integer (nullable = true)



In [7]:
range_check_row = df_check.select(
    F.sum(F.when(F.col("Sales").isNull(), 1).otherwise(0)).alias("Sales null/sai kiểu"),
    F.sum(F.when(F.col("Profit").isNull(), 1).otherwise(0)).alias("Profit null/sai kiểu"),
    F.sum(F.when(F.col("Quantity").isNull(), 1).otherwise(0)).alias("Quantity null/sai kiểu"),
    F.sum(F.when(F.col("Discount").isNull(), 1).otherwise(0)).alias("Discount null/sai kiểu"),
    F.sum(F.when(F.col("Shipping_Cost").isNull(), 1).otherwise(0)).alias("Shipping_Cost null/sai kiểu"),
    F.sum(F.when(F.col("Sales") <= 0, 1).otherwise(0)).alias("Sales <= 0"),
    F.sum(F.when(F.col("Quantity") < 1, 1).otherwise(0)).alias("Quantity < 1"),
    F.sum(F.when((F.col("Discount") < 0) | (F.col("Discount") > 0.85), 1).otherwise(0)).alias("Discount ngoài khoảng 0-0.85"),
    F.sum(F.when(F.col("Shipping_Cost") < 0, 1).otherwise(0)).alias("Shipping_Cost < 0"),
    F.sum(F.when((F.col("Year") < 2011) | (F.col("Year") > 2014), 1).otherwise(0)).alias("Year ngoài 2011-2014"),
    F.sum(F.when((F.col("weeknum") < 1) | (F.col("weeknum") > 53), 1).otherwise(0)).alias("weeknum ngoài 1-53")
).first().asDict()

range_check_df = pd.DataFrame(
    list(range_check_row.items()),
    columns=["Tiêu chí kiểm tra", "Số dòng vi phạm"]
)

display(range_check_df)

range_violation_total = sum(range_check_row.values())

print("Tổng số vi phạm miền giá trị:", range_violation_total)

,Tiêu chí kiểm tra,Số dòng vi phạm
0,Sales null/sai kiểu,0
1,Profit null/sai kiểu,0
2,Quantity null/sai kiểu,0
3,Discount null/sai kiểu,0
4,Shipping_Cost null/sai kiểu,0
5,Sales <= 0,0
6,Quantity < 1,0
7,Discount ngoài khoảng 0-0.85,0
8,Shipping_Cost < 0,0
9,Year ngoài 2011-2014,0


Tổng số vi phạm miền giá trị: 0


In [8]:
df_time = (
    df_check
    .withColumn("Order_Date_dt", F.to_date("Order_Date"))
    .withColumn("Ship_Date_dt", F.to_date("Ship_Date"))
)

time_check_row = df_time.select(
    F.sum(F.when(F.col("Order_Date_dt").isNull(), 1).otherwise(0)).alias("Order_Date không đọc được"),
    F.sum(F.when(F.col("Ship_Date_dt").isNull(), 1).otherwise(0)).alias("Ship_Date không đọc được"),
    F.sum(F.when(F.col("Ship_Date_dt") < F.col("Order_Date_dt"), 1).otherwise(0)).alias("Ship_Date < Order_Date"),
    F.sum(
        F.when(
            (F.col("Order_Date_dt") < F.lit("2011-01-01").cast("date")) |
            (F.col("Order_Date_dt") > F.lit("2014-12-31").cast("date")),
            1
        ).otherwise(0)
    ).alias("Order_Date ngoài 2011-2014")
).first().asDict()

time_check_df = pd.DataFrame(
    list(time_check_row.items()),
    columns=["Tiêu chí kiểm tra", "Số dòng vi phạm"]
)

display(time_check_df)

time_violation_total = sum(time_check_row.values())

print("Tổng số vi phạm thời gian:", time_violation_total)

,Tiêu chí kiểm tra,Số dòng vi phạm
0,Order_Date không đọc được,0
1,Ship_Date không đọc được,0
2,Ship_Date < Order_Date,0
3,Order_Date ngoài 2011-2014,0


Tổng số vi phạm thời gian: 0


In [9]:
total_rows = df_check.count()
total_customers = df_check.select("Customer_ID").distinct().count()
total_orders = df_check.select("Order_ID").distinct().count()
total_products = df_check.select("Product_ID").distinct().count()

lines_per_order = df_check.groupBy("Order_ID").count()

orders_per_customer = df_check.groupBy("Customer_ID").agg(
    F.countDistinct("Order_ID").alias("n_orders")
)

avg_lines_per_order = lines_per_order.agg(F.avg("count")).first()[0]
avg_orders_per_customer = orders_per_customer.agg(F.avg("n_orders")).first()[0]

structure_summary_df = pd.DataFrame(
    [
        ("Tổng số dòng giao dịch", total_rows),
        ("Số khách hàng", total_customers),
        ("Số đơn hàng", total_orders),
        ("Số sản phẩm", total_products),
        ("Số dòng trung bình / đơn hàng", round(avg_lines_per_order, 2)),
        ("Số đơn hàng trung bình / khách hàng", round(avg_orders_per_customer, 2)),
    ],
    columns=["Tiêu chí", "Kết quả"]
)

display(structure_summary_df)

,Tiêu chí,Kết quả
0,Tổng số dòng giao dịch,500000.00
1,Số khách hàng,33313.00
2,Số đơn hàng,236373.00
3,Số sản phẩm,10292.00
4,Số dòng trung bình / đơn hàng,2.12
5,Số đơn hàng trung bình / khách hàng,7.12


In [10]:
final_validation_df = pd.DataFrame(
    [
        (
            "Quy mô dữ liệu",
            "500.000 dòng",
            "Đạt" if row_count == 500000 else "Cần kiểm tra"
        ),
        (
            "Schema",
            "Đúng 24 cột, không thiếu/không dư cột",
            "Đạt" if len(missing_columns) == 0 and len(extra_columns) == 0 else "Cần kiểm tra"
        ),
        (
            "Missing value",
            "Không có missing/rỗng",
            "Đạt" if total_missing == 0 else "Cần kiểm tra"
        ),
        (
            "Duplicate",
            "Không có dòng trùng hoàn toàn",
            "Đạt" if duplicate_rows == 0 else "Cần kiểm tra"
        ),
        (
            "Miền giá trị",
            "Không vi phạm ràng buộc chính",
            "Đạt" if range_violation_total == 0 else "Cần kiểm tra"
        ),
        (
            "Thời gian",
            "Không có lỗi ngày tháng cơ bản",
            "Đạt" if time_violation_total == 0 else "Cần kiểm tra"
        ),
        (
            "Cấu trúc giao dịch",
            "Có cấu trúc khách hàng - đơn hàng - dòng sản phẩm",
            "Đạt" if total_customers > 0 and total_orders > 0 and avg_lines_per_order >= 1 else "Cần kiểm tra"
        ),
    ],
    columns=["Nhóm kiểm tra", "Kết quả mong muốn", "Trạng thái"]
)

display(final_validation_df)

,Nhóm kiểm tra,Kết quả mong muốn,Trạng thái
0,Quy mô dữ liệu,500.000 dòng,Đạt
1,Schema,"Đúng 24 cột, không thiếu/không dư cột",Đạt
2,Missing value,Không có missing/rỗng,Đạt
3,Duplicate,Không có dòng trùng hoàn toàn,Đạt
4,Miền giá trị,Không vi phạm ràng buộc chính,Đạt
5,Thời gian,Không có lỗi ngày tháng cơ bản,Đạt
6,Cấu trúc giao dịch,Có cấu trúc khách hàng - đơn hàng - dòng sản phẩm,Đạt
